# 07. Output Guardrail (출력 가드레일)

**01_Agents_Overview**에서 `@input_guardrail`로 **입력**을 검증하는 방법을 배웠습니다.
이번에는 `@output_guardrail`로 에이전트의 **출력**을 검증하는 방법을 알아봅니다.

### Input vs Output Guardrail 비교

| 구분 | Input Guardrail | Output Guardrail |
|------|----------------|-----------------|
| 검증 시점 | 에이전트 실행 **전** | 에이전트 실행 **후** |
| 검증 대상 | 사용자 입력 | 에이전트 출력 |
| 데코레이터 | `@input_guardrail` | `@output_guardrail` |
| 예외 | `InputGuardrailTripwireTriggered` | `OutputGuardrailTripwireTriggered` |
| 실행 모드 | Parallel(병렬) 또는 Blocking(차단) | 항상 Sequential(순차) |

### Tripwire (트립와이어)란?
가드레일이 문제를 감지했을 때 발생시키는 **즉시 중단 신호**입니다.
`tripwire_triggered=True`로 설정하면 즉시 예외가 발생하고 실행이 중단됩니다.

## 1. 기본 Output Guardrail

에이전트의 출력에 개인정보(이메일, 전화번호)가 포함되어 있는지 검사하는 가드레일을 만듭니다.

In [ ]:
# 출력 가드레일 정의: 개인정보(PII) 검출
    # 이메일 패턴 검사
    # 전화번호 패턴 검사 (한국 형식)

## 2. 가드레일을 에이전트에 등록

In [ ]:
# 에이전트에 output_guardrails 등록

## 3. 정상 출력 테스트

PII가 포함되지 않는 일반 질문으로 테스트합니다.

In [ ]:
# 정상적인 응답 (PII 없음) - 통과해야 함

## 4. Tripwire 발동 테스트

에이전트가 PII를 포함한 응답을 생성하도록 유도하여 가드레일이 차단하는지 테스트합니다.

In [ ]:
# Tripwire 발동 테스트를 위해 PII를 포함하도록 지시하는 에이전트 생성
# PII가 포함된 응답을 유도 - Tripwire 발동해야 함

## 5. LLM 기반 Output Guardrail

정규식 대신 **별도의 경량 LLM**을 사용하여 출력을 검증하는 고급 패턴입니다.
검증용 에이전트가 출력의 적절성을 판단합니다.

In [ ]:
# ---------------------------------------
# Guardrail 검증 결과를 담기 위한 Pydantic 모델
# ---------------------------------------
class GuardrailCheckResult(BaseModel):
# ---------------------------------------
# LLM 기반 출력 검증 전용 에이전트
# ---------------------------------------
# 실제 서비스에서는 비용 절감을 위해 작은 모델을 사용하는 경우가 많음
    # 검증 기준을 명확하게 지시
    # LLM 출력 결과를 Pydantic 모델로 강제 구조화
# ---------------------------------------
# Output Guardrail 함수 정의
# ---------------------------------------
# 에이전트가 응답을 생성한 후 실행되는 검증 함수
    # 검증 전용 에이전트를 실행하여 응답 검증
    # 검증 결과를 GuardrailFunctionOutput 형식으로 반환
        # 추가 정보 (로깅/모니터링용)
        # True이면 가드레일 트리거 발생 → 응답 차단
# ---------------------------------------
# Guardrail이 적용된 메인 에이전트
# ---------------------------------------
    # 기본 역할 정의
    # 출력 생성 후 실행될 Guardrail 목록

In [ ]:
# 정상적인 질문 테스트

In [ ]:
# 부적절한 응답을 유도하는 테스트 - Tripwire 발동해야 함

### 정리

| 개념 | 설명 |
|------|------|
| `@output_guardrail` | 에이전트 출력을 검증하는 데코레이터 |
| `GuardrailFunctionOutput` | 가드레일 검증 결과 (output_info + tripwire_triggered) |
| `tripwire_triggered=True` | 즉시 예외 발생 → 실행 중단 |
| `OutputGuardrailTripwireTriggered` | 트립와이어 발동 시 발생하는 예외 |
| `output_guardrails=[...]` | 에이전트에 가드레일 등록 |
| LLM 기반 가드레일 | 별도의 검증 에이전트를 사용하여 출력 적절성 판단 |

**활용 사례:**
- 개인정보(PII) 유출 방지
- 부적절한 콘텐츠 필터링
- 응답 품질 검증 (할루시네이션 탐지)
- 규정 준수 확인 (금융, 의료 등)